# Vibe Training, Live

A tiny version of Karpathy's autoresearch loop, runnable on a laptop in ~3 minutes.

**The setup.** We have a small MLP learning a regression task with a deliberately bad baseline config. Instead of *us* tweaking learning rates and watching loss, we hand the loop to Claude. Claude:

1. Reads the current config and the loss history
2. Proposes **one** change
3. We run the experiment, report back the val loss
4. Claude decides: keep, revert, or pivot — and proposes the next change

This is exactly the structure Karpathy's `autoresearch` agent uses, just shrunk small enough to demo on camera. The lesson lands the same way: **the loop is easy, the reward is everything.**

## Part 1 — The training script we're handing to Claude

We're keeping the action space tiny on purpose. Slide 18 of the deck said: "cap the action space, don't let one agent change 700 things." That's exactly what we do here.

Five knobs only:
- `lr` — learning rate
- `hidden_dim` — width of the MLP
- `n_layers` — depth
- `weight_decay` — L2 regularisation
- `batch_size`

The reward function is a single number: **validation MSE after 200 training steps**. Lower is better. That's our verifiable reward.

In [ ]:
import torch, torch.nn as nn, numpy as np, random, json, copy
from typing import Dict, List

# Reproducibility — important so that "the gain Claude found" is real, not noise.
def set_seeds(s=42):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)

# A small synthetic non-linear regression task.
def make_data(n=2000, d=8, seed=0):
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, d)).astype(np.float32)
    # Target depends on a non-linear interaction of 3 features + noise.
    y = (np.sin(X[:,0]) * X[:,1] + 0.5 * X[:,2]**2 + 0.1 * rng.standard_normal(n)).astype(np.float32)
    split = int(0.8 * n)
    return (torch.from_numpy(X[:split]), torch.from_numpy(y[:split])), (torch.from_numpy(X[split:]), torch.from_numpy(y[split:]))

(Xtr, ytr), (Xva, yva) = make_data()
print("train:", Xtr.shape, "val:", Xva.shape)

In [ ]:
# The training run. This is what Claude is going to optimise.
# Returns the final validation MSE as a single scalar reward.

def train_once(config: Dict) -> float:
    set_seeds(42)  # same seed every run -> any loss change is from the config, not RNG
    layers = []
    in_d = Xtr.shape[1]
    for _ in range(config["n_layers"]):
        layers += [nn.Linear(in_d, config["hidden_dim"]), nn.ReLU()]
        in_d = config["hidden_dim"]
    layers += [nn.Linear(in_d, 1)]
    model = nn.Sequential(*layers)

    opt = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    loss_fn = nn.MSELoss()

    n_steps = 200
    bs = config["batch_size"]
    for step in range(n_steps):
        idx = torch.randint(0, len(Xtr), (bs,))
        pred = model(Xtr[idx]).squeeze(-1)
        loss = loss_fn(pred, ytr[idx])
        opt.zero_grad(); loss.backward(); opt.step()

    with torch.no_grad():
        val_pred = model(Xva).squeeze(-1)
        val_mse = loss_fn(val_pred, yva).item()
    return round(val_mse, 4)

## Part 2 — The deliberately bad baseline

We start with a config that's *almost* OK but has obvious problems. Watch the val loss:

In [ ]:
baseline_config = {
    "lr": 0.5,            # way too high
    "hidden_dim": 4,      # too narrow for this task
    "n_layers": 1,        # too shallow
    "weight_decay": 0.0,
    "batch_size": 8,      # noisy
}

baseline_loss = train_once(baseline_config)
print(f"Baseline val MSE: {baseline_loss}")

## Part 3 — One round of "human vibe training" first

Before letting Claude do it, let's see what *we* would do. The lr is obviously too high. Drop it:

In [ ]:
human_try = {**baseline_config, "lr": 0.01}
human_loss = train_once(human_try)
print(f"After we lower lr from 0.5 -> 0.01: val MSE = {human_loss}")
print(f"Delta vs baseline: {human_loss - baseline_loss:+.4f}")

That's "vibe coding" energy applied to training: we made one change because we *felt* it was right, ran it, looked at the number. Now imagine we have to do this 50 times across 5 dimensions. That's exactly the loop we're about to hand to Claude.

## Part 4 — Wire up Claude as the autoresearch agent

> **Before running this cell:** export your Anthropic API key in your shell:
> ```bash
> export ANTHROPIC_API_KEY=sk-ant-...
> ```
> Get one at [console.anthropic.com](https://console.anthropic.com). The whole 8-round demo costs roughly a cent on `claude-opus-4-7`.

We give Claude:
- The current config and what we've tried before (history)
- The reward function description (lower val MSE is better)
- A strict output format: one JSON object with `reasoning`, `change`, and `expected_direction`

The strict format is doing the same job as RLVR — it makes Claude's output **verifiable**: we can parse it, run it, and grade the result without human judgement in the loop.

In [ ]:
import os
from anthropic import Anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "Set ANTHROPIC_API_KEY in your environment before running this cell. "
        "See https://console.anthropic.com to get a key."
    )

client = Anthropic()
MODEL = "claude-opus-4-7"   # the current Opus model

SYSTEM_PROMPT = '''You are an autoresearch agent tuning a tiny MLP for regression.

The training script is fixed. You can only modify these 5 hyperparameters:
  - lr            (float, 1e-5 to 1.0)
  - hidden_dim    (int, 2 to 256)
  - n_layers      (int, 1 to 6)
  - weight_decay  (float, 0.0 to 0.1)
  - batch_size    (int, 4 to 256)

The reward is validation MSE after 200 training steps. LOWER is better.

On every turn you will see:
  - The current config
  - The history of (config, val_mse) tuples you have tried
  - Your previous reasoning

You must propose exactly ONE hyperparameter change per turn. Constrain the action space.
If a previous change made things worse, REVERT it before exploring a new direction.

Respond with ONLY a JSON object, no prose, in this exact format:
{
  "reasoning": "<one sentence on what the history tells you and why this change>",
  "change": {"<param_name>": <new_value>},
  "expected_direction": "lower" | "higher" | "uncertain"
}
'''
print("Agent wired up. Model:", MODEL)

In [ ]:
# The agent step: build the prompt from history, call Claude, parse the JSON.
import re

def extract_json(text: str) -> dict:
    """Robustly pull a JSON object out of Claude's response."""
    # Strip ```json ... ``` or ``` ... ``` fences if present.
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    # If there's any prose before/after the JSON, grab the first {...} block.
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if m:
        text = m.group(0)
    return json.loads(text)

def ask_claude_for_next_change(current_config: Dict, history: List[Dict]) -> Dict:
    history_text = json.dumps(history, indent=2) if history else "(none yet — this is the first move)"
    user_msg = f'''Current config:
{json.dumps(current_config, indent=2)}

History so far:
{history_text}

What is your next single-parameter change?'''

    resp = client.messages.create(
        model=MODEL,
        max_tokens=400,
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_msg}],
    )
    return extract_json(resp.content[0].text)

## Part 5 — Run the loop

We'll let Claude run **8 rounds**. On camera, you can talk through what each round means: what Claude is keeping, what it's reverting, where it's exploring.

Watch for the moment Claude does its own credit assignment — it'll say something like *"the previous lr drop helped, lr is now in a good range, switching to width."* That sentence is the whole video in one line.

In [ ]:
def run_vibe_training(rounds: int = 8):
    config = dict(baseline_config)        # start from the bad baseline
    best_loss = train_once(config)
    best_config = dict(config)
    history = [{"config": dict(config), "val_mse": best_loss, "note": "baseline"}]

    print(f"Round 0 (baseline): val_mse = {best_loss}")
    print("-" * 60)

    for r in range(1, rounds + 1):
        # 1) Ask Claude
        proposal = ask_claude_for_next_change(config, history)
        param, new_val = list(proposal["change"].items())[0]

        # 2) Apply
        prev_val = config[param]
        config[param] = new_val

        # 3) Run + grade (verifiable reward)
        val_mse = train_once(config)
        delta = val_mse - history[-1]["val_mse"]
        kept = val_mse <= best_loss
        if kept:
            best_loss = val_mse
            best_config = dict(config)

        print(f"Round {r}:")
        print(f"  reasoning : {proposal['reasoning']}")
        print(f"  change    : {param}: {prev_val} -> {new_val}")
        print(f"  val_mse   : {val_mse}  (delta {delta:+.4f})  {'KEPT' if kept else 'WORSE — reverting'}")

        # 4) Revert if worse — this is the keep/revert logic from autoresearch
        if not kept:
            config[param] = prev_val

        history.append({
            "config": dict(config),
            "val_mse": val_mse,
            "note": f"{param}: {prev_val} -> {new_val} ({'kept' if kept else 'reverted'})"
        })
        print("-" * 60)

    return history, best_loss, best_config

In [ ]:
history, best_loss, best_config = run_vibe_training(rounds=8)

print("\n" + "=" * 60)
print("RESULT")
print("=" * 60)
print(f"Baseline val_mse : {history[0]['val_mse']}")
print(f"Best val_mse     : {best_loss}")
print(f"Improvement      : {(history[0]['val_mse'] - best_loss):.4f}  "
      f"({100 * (history[0]['val_mse'] - best_loss) / history[0]['val_mse']:.1f}% lower)")
print(f"Best config      : {json.dumps(best_config, indent=2)}")

## Part 6 — Plot the loss curve

Show this on camera. Karpathy's tweet went viral because it was a real, monotonic-ish drop. Yours will look similar — and more importantly, you can point at each point and say *which* hyperparameter Claude was changing.

In [ ]:
import matplotlib.pyplot as plt

losses = [h["val_mse"] for h in history]
notes  = [h["note"] for h in history]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(len(losses)), losses, marker="o", color="#0B1437", linewidth=2)
ax.axhline(losses[0], linestyle="--", color="#F96167", alpha=0.6, label="baseline")
ax.set_xlabel("round")
ax.set_ylabel("validation MSE")
ax.set_title("Vibe training: Claude tuning a tiny MLP")
ax.grid(True, alpha=0.2)
ax.legend()

# annotate each round with what changed
for i, n in enumerate(notes):
    if i == 0: continue
    ax.annotate(n.split(" (")[0], (i, losses[i]),
                fontsize=8, xytext=(5, 5), textcoords="offset points", alpha=0.7)

plt.tight_layout()
plt.show()

## Part 7 — The credit-assignment moment

Print the trace one more time, but now grouped: which changes were KEPT (helped) vs REVERTED (didn't). This is the slide-12 "credit assignment" content made concrete.

In [ ]:
kept    = [h for h in history[1:] if "kept" in h["note"]]
reverted = [h for h in history[1:] if "reverted" in h["note"]]

print(f"KEPT ({len(kept)}):")
for h in kept:
    print(f"   ✓ {h['note']}  ->  val_mse {h['val_mse']}")

print(f"\nREVERTED ({len(reverted)}):")
for h in reverted:
    print(f"   ✗ {h['note']}  ->  val_mse {h['val_mse']}")

print()
print("This is the part Karpathy's full-scale run can't easily do —")
print("with 700 changes, you can't hand-trace what mattered. With 8, you can.")

## Part 8 — What this demo proves and what it doesn't

**Proves:**
- The autoresearch loop works at any scale. You can hand Claude a verifiable reward and a constrained action space and it will improve a model.
- The reasoning per round is genuinely useful — Claude tells you *why* it thinks a change should help. That's the credit assignment we lose at 700-change scale.
- Reverting on regression is what makes vibe training safe. Without it, the loop drifts.

**Doesn't prove:**
- That this works on a real frontier model run. Hyperparameter tuning a tiny MLP is the easy regime.
- That the reward is automatically a good one. We chose val MSE because it's verifiable. On a real product, "is this model good?" rarely reduces to one scalar.
- That you should fire your ML engineers. The whole reason this loop runs is that *we* defined the action space, the reward, and the budget. **That's the reward design from slide 15.** The agent is doing search; we're doing the thinking.

> The loop is easy. The reward is everything.

— that's your closing line on camera.